# Maverick Programme: Total Portfolio Incrementality

## Why portfolio measurement?

Flash Sales in Dortmund/Dresden/Essen cannot be measured in isolation because 6+ concurrent Maverick campaigns run simultaneously:
- Smart Growth (Engaged), SVP (New/Reactivated), Win Back (Churned/Dormant/Lapsing)
- TFD (All), Prospect €15, Offline vouchering, Refer a friend, Sales Enablement

Any city-level BSTS comparison captures the **combined effect** of all programmes, not just Flash Sales. This notebook measures that total portfolio effect honestly.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import time, gc, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)

import aa_config as cfg
import flash_sales_utils as fsu

client = bigquery.Client(project=cfg.PROJECT_ID)

# === PORTFOLIO CONFIGURATION ===
COUNTRY = 'DE'
TREATMENT_CITIES = ['Dortmund', 'Dresden', 'Essen']
# Full Maverick programme period (earliest campaign start to latest end)
PORTFOLIO_START = '2026-02-04'  # TFD launch
PORTFOLIO_END = '2026-03-31'    # Most campaigns end
# Pre-period: 12 weeks before programme start (no Maverick activity)
PRE_START = '2025-11-10'
PRE_END = '2026-02-03'

print(f'Portfolio period: {PORTFOLIO_START} to {PORTFOLIO_END}')
print(f'Pre-period: {PRE_START} to {PRE_END}')
print(f'Treatment cities: {TREATMENT_CITIES}')

## 2. Campaign Timeline Visualization

In [ ]:
for city in TREATMENT_CITIES:
    fig = fsu.plot_campaign_timeline(
        city,
        date_range_start='2026-01-01',
        date_range_end='2026-04-07',
        highlight_dates=[(PORTFOLIO_START, PORTFOLIO_END)])
    fig.suptitle(f'{city}: All Maverick Activity', y=1.02, fontsize=14)
    fig.show()

## 3. Control City Selection (MAVERICK)

In [ ]:
# Use pre-period end as reference date for city features
df_cities = fsu.get_city_features(client, COUNTRY, PRE_END)
print(f'{len(df_cities)} cities with features')

# Find controls for each treatment city
city_controls = {}
for city in TREATMENT_CITIES:
    controls = fsu.find_control_cities(df_cities, city, cfg.FLASH_SALES_KNN_NEIGHBORS)
    print(f'\n{city}: {len(controls)} KNN candidates')
    city_controls[city] = controls

# Shared controls = intersection of all cities' control pools
shared = set(city_controls[TREATMENT_CITIES[0]])
for city in TREATMENT_CITIES[1:]:
    shared &= set(city_controls[city])
print(f'\nShared controls across all 3 cities: {len(shared)}')
print(sorted(shared))

## 4. Daily Data & Correlation Filter

In [ ]:
# For portfolio measurement, use daily granularity (not hourly)
# because the campaigns run all day, not just dinner hours
results_per_city = {}

for city in TREATMENT_CITIES:
    print(f'\n=== {city} ===')
    controls = city_controls[city]
    all_cities = [city] + controls

    # Pull daily orders for full pre + post period
    df_orders = fsu.get_daily_orders(
        client, COUNTRY, all_cities, PRE_START, PORTFOLIO_END)
    print(f'  Daily orders: {len(df_orders)} rows')

    # Correlation filter on pre-period
    df_pre = df_orders[
        pd.to_datetime(df_orders['orderdate']) <= pd.to_datetime(PRE_END)
    ]
    filtered = fsu.apply_correlation_filter(
        df_pre, city, controls,
        cfg.FLASH_SALES_CORRELATION_THRESHOLD,
        time_col='orderdate')

    if not filtered:
        print(f'  No controls pass correlation filter for {city}')
        continue

    # Contamination check on control cities
    contam = fsu.check_control_contamination(
        client, COUNTRY, filtered, PORTFOLIO_START, PORTFOLIO_END)
    contaminated = contam['city'].unique().tolist() if len(contam) > 0 else []
    clean = [c for c in filtered if c not in contaminated]
    print(f'  Clean controls: {len(clean)}/{len(filtered)}')

    results_per_city[city] = {
        'df_orders': df_orders,
        'filtered_controls': filtered,
        'clean_controls': clean,
        'contaminated': contaminated,
    }

## 5. CausalImpact — Per City

In [ ]:
city_results = {}

for city in TREATMENT_CITIES:
    if city not in results_per_city:
        continue
    info = results_per_city[city]
    controls = info['clean_controls'] or info['filtered_controls']
    df_orders = info['df_orders']
    print(f'\n=== CausalImpact: {city} ({len(controls)} controls) ===')

    # Build daily pivot
    all_cities = [city] + controls
    df = df_orders[df_orders['city'].isin(all_cities)]
    pivot = df.pivot_table(
        index='orderdate', columns='city', values='totalorders'
    ).fillna(0).sort_index()
    pivot.index = pd.to_datetime(pivot.index)
    cols = [city] + [c for c in pivot.columns if c != city]
    pivot = pivot[cols]

    pre_period = [pd.to_datetime(PRE_START), pd.to_datetime(PRE_END)]
    post_period = [pd.to_datetime(PORTFOLIO_START), pd.to_datetime(PORTFOLIO_END)]

    result = fsu.run_causal_impact(pivot, pre_period, post_period)
    if result:
        incr = result['actual_orders'] - result['predicted_orders']
        print(f'  Actual:      {result["actual_orders"]:.0f}')
        print(f'  Predicted:   {result["predicted_orders"]:.0f}')
        print(f'  Incremental: {incr:+.0f}')
        print(f'  Uplift:      {result["uplift"]:+.4f} ({result["uplift"]*100:+.1f}%)')
        city_results[city] = result
    else:
        print(f'  CausalImpact failed for {city}')
    gc.collect()

## 6. Combined Results

In [ ]:
print('=' * 60)
print('MAVERICK PORTFOLIO: TOTAL INCREMENTALITY')
print('=' * 60)
print(f'Period: {PORTFOLIO_START} to {PORTFOLIO_END}')
print(f'Pre-period: {PRE_START} to {PRE_END}')
print()

total_actual = 0
total_predicted = 0

for city in TREATMENT_CITIES:
    if city not in city_results:
        print(f'{city}: No result')
        continue
    r = city_results[city]
    incr = r['actual_orders'] - r['predicted_orders']
    total_actual += r['actual_orders']
    total_predicted += r['predicted_orders']
    info = results_per_city[city]
    n_clean = len(info['clean_controls'])
    n_contam = len(info['contaminated'])
    print(f'{city}:')
    print(f'  Uplift: {r["uplift"]:+.4f} ({r["uplift"]*100:+.1f}%)')
    print(f'  Incremental orders: {incr:+.0f}')
    print(f'  Controls: {n_clean} clean, {n_contam} contaminated')
    print()

if total_predicted > 0:
    total_incr = total_actual - total_predicted
    total_uplift = total_actual / total_predicted - 1
    print(f'COMBINED (all 3 cities):')
    print(f'  Total actual orders:      {total_actual:.0f}')
    print(f'  Total predicted orders:   {total_predicted:.0f}')
    print(f'  Total incremental orders: {total_incr:+.0f}')
    print(f'  Portfolio uplift:         {total_uplift:+.4f} ({total_uplift*100:+.1f}%)')

print()
print('NOTE: This captures the COMBINED effect of ALL Maverick')
print('campaigns (Smart Growth, SVP, Win Back, TFD, Flash Sales,')
print('Prospect, Refer a Friend, Sales Enablement, Offline vouchering).')
print('It CANNOT be attributed to Flash Sales alone.')
print('=' * 60)

## 7. Caveats

1. **This is a portfolio measurement.** The uplift includes Smart Growth, SVP, Win Back, Flash Sales, TFD, and all other Maverick programmes combined.
2. **Control city contamination** is checked via `fact_offer` but other non-OMT campaigns (national promos, brand campaigns) may still affect controls.
3. **Pre-period includes Christmas/NYE.** The 12-week pre-period (Nov 10 - Feb 3) spans the holiday season. BSTS should handle this via covariates, but holiday effects in treatment cities may differ from controls.
4. **No control for Maverick-specific selection effects.** Maverick cities were chosen because they are expansion markets — they may have inherently different growth trajectories than control cities.